In [2]:
"""
CarbonTatva AI — Model: CNN-LSTM (Speed Optimized)
====================================================
Predicts next-day (96 slots = 24h) electricity prices
separately for GDAM, DAM, RTM markets.

Optimized for Mac MPS / Windows CPU — trains in reasonable time.

Run:
    python model_cnnlstm.py

Output: outputs_cnnlstm/
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
os.makedirs('outputs_cnnlstm', exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# DEVICE SELECTION — Mac MPS / CUDA / CPU
# ─────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"  Device: CUDA ({torch.cuda.get_device_name(0)})")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print("  Device: MPS (Apple Silicon GPU)")
    else:
        device = torch.device('cpu')
        print("  Device: CPU")
    return device

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE  = 'final_merged_data2.xlsx'
MARKETS    = ['GDAM', 'DAM', 'RTM']

SLOTS_DAY  = 96
LOOKBACK   = SLOTS_DAY * 1   # 1 day lookback (was 3 — 3x faster)
HORIZON    = SLOTS_DAY
TRAIN_END  = '2024-12-31'
VAL_END    = '2025-09-30'
SEED       = 42
BATCH_SIZE = 128              # was 64 — fewer steps per epoch
EPOCHS     = 50               # was 80 — early stopping handles this

torch.manual_seed(SEED)
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()

    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend']   = (df['dow'] >= 5).astype(int)
    df['is_evening']   = ((df['hour'] >= 18) & (df['hour'] <= 21)).astype(int)
    df['is_solar_hrs'] = ((df['hour'] >= 8)  & (df['hour'] <= 16)).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    p = df['Price (Rs./MWh)']
    df['log_price']    = np.log1p(p)
    df['price_lag96']  = p.shift(96)
    df['price_lag672'] = p.shift(672)
    df['ma3d'] = p.shift(1).rolling(288).mean()
    df['ma7d'] = p.shift(1).rolling(672).mean()
    df['price_diff1d'] = p.shift(96)  - p.shift(192)
    df['price_diff1w'] = p.shift(672) - p.shift(1344)

    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']

    return df


PAST_FEATURES = [
    'log_price',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'radiation', 'temp', 'wind_speed',
    'cloud_cover', 'is_weekend',
]

FUTURE_FEATURES = [
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    'is_evening', 'is_solar_hrs',
    'price_lag96', 'price_lag672',
    'ma3d', 'ma7d',
    'price_diff1d', 'price_diff1w',
    'radiation', 'radiation_x_hour_sin',
    'temp', 'temp_sq',
    'wind_speed', 'cloud_cover', 'humidity',
]

N_PAST   = len(PAST_FEATURES)
N_FUTURE = len(FUTURE_FEATURES)

# ─────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────
class ElectricityDataset(Dataset):
    def __init__(self, df, lookback=LOOKBACK, horizon=HORIZON):
        all_feats = list(dict.fromkeys(PAST_FEATURES + FUTURE_FEATURES))
        df = df.dropna(subset=all_feats + ['log_price']).reset_index(drop=True)
        self.past_data   = df[PAST_FEATURES].values.astype(np.float32)
        self.future_data = df[FUTURE_FEATURES].values.astype(np.float32)
        self.log_prices  = df['log_price'].values.astype(np.float32)
        self.lookback    = lookback
        self.horizon     = horizon

    def __len__(self):
        return len(self.log_prices) - self.lookback - self.horizon + 1

    def __getitem__(self, idx):
        past_end   = idx + self.lookback
        future_end = past_end + self.horizon
        return (torch.from_numpy(self.past_data[idx:past_end]),
                torch.from_numpy(self.future_data[past_end:future_end]),
                torch.from_numpy(self.log_prices[past_end:future_end]))


# ─────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        weights = torch.softmax(self.attn(lstm_out).squeeze(-1), dim=1)
        return (lstm_out * weights.unsqueeze(-1)).sum(dim=1)


class CNNLSTM(nn.Module):
    def __init__(self, n_past, n_future,
                 cnn_filters=64,
                 lstm_hidden=128,
                 lstm_layers=2,
                 horizon=HORIZON,
                 dropout=0.2):
        super().__init__()
        self.horizon = horizon

        self.cnn3 = nn.Sequential(
            nn.Conv1d(n_past, cnn_filters, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_filters), nn.GELU())
        self.cnn5 = nn.Sequential(
            nn.Conv1d(n_past, cnn_filters, kernel_size=5, padding=2),
            nn.BatchNorm1d(cnn_filters), nn.GELU())
        self.cnn7 = nn.Sequential(
            nn.Conv1d(n_past, cnn_filters, kernel_size=7, padding=3),
            nn.BatchNorm1d(cnn_filters), nn.GELU())

        cnn_out_size  = cnn_filters * 3
        self.lstm = nn.LSTM(cnn_out_size, lstm_hidden, lstm_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if lstm_layers > 1 else 0)
        lstm_out_size = lstm_hidden * 2

        self.attention = AttentionLayer(lstm_out_size)

        self.future_proj = nn.Sequential(
            nn.Linear(n_future, 32), nn.LayerNorm(32), nn.GELU())

        fusion_in = lstm_out_size + 32 * horizon
        self.output_net = nn.Sequential(
            nn.Linear(fusion_in, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, horizon))

    def forward(self, x_past, x_future):
        B   = x_past.shape[0]
        xp  = x_past.permute(0, 2, 1)
        cnn = torch.cat([self.cnn3(xp), self.cnn5(xp),
                         self.cnn7(xp)], dim=1).permute(0, 2, 1)
        ctx = self.attention(self.lstm(cnn)[0])
        fut = self.future_proj(x_future).reshape(B, -1)
        return self.output_net(torch.cat([ctx, fut], dim=1))


# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:20s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#2980b9', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 3
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#2980b9', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 3 Days Zoomed', color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#2980b9')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted Scatter', color='white', fontsize=12)
    axes[2].set_xlabel('Actual Price', color='#aaa')
    axes[2].set_ylabel('Predicted Price', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


# ─────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: CNN-LSTM (Speed Optimized)")
print("=" * 60)
DEVICE = get_device()

print(f"Loading data from {DATA_FILE}...")
sheets    = pd.read_excel(DATA_FILE, sheet_name=None)
all_feats = list(dict.fromkeys(PAST_FEATURES + FUTURE_FEATURES))
all_metrics = {}

for mkt in MARKETS:
    print(f"\n{'─'*50}")
    print(f"Market: {mkt}")
    print('─' * 50)

    df       = build_features(sheets[mkt])
    df_clean = df.dropna(subset=all_feats + ['log_price']).copy()

    train_df = df_clean[df_clean['datetime'] <= TRAIN_END]
    val_df   = df_clean[(df_clean['datetime'] > TRAIN_END) &
                         (df_clean['datetime'] <= VAL_END)]
    test_df  = df_clean[df_clean['datetime'] > VAL_END]
    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

    scaler  = StandardScaler()
    train_s = train_df.copy()
    val_s   = val_df.copy()
    test_s  = test_df.copy()
    train_s[all_feats] = scaler.fit_transform(train_df[all_feats])
    val_s[all_feats]   = scaler.transform(val_df[all_feats])
    test_s[all_feats]  = scaler.transform(test_df[all_feats])
    train_s['log_price'] = train_df['log_price'].values
    val_s['log_price']   = val_df['log_price'].values
    test_s['log_price']  = test_df['log_price'].values

    train_ds = ElectricityDataset(train_s, LOOKBACK, HORIZON)
    val_ds   = ElectricityDataset(val_s,   LOOKBACK, HORIZON)
    test_ds  = ElectricityDataset(test_s,  LOOKBACK, HORIZON)
    print(f"  Samples — Train: {len(train_ds):,} | "
          f"Val: {len(val_ds):,} | Test: {len(test_ds):,}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)

    model = CNNLSTM(N_PAST, N_FUTURE, 64, 128, 2, HORIZON, 0.2).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    def lr_lambda(epoch):
        WARMUP = 3
        if epoch < WARMUP:
            return epoch / WARMUP
        return 0.5 * (1 + np.cos(np.pi * (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.HuberLoss(delta=1.5)

    PATIENCE  = 8
    best_val  = float('inf')
    pat_count = 0
    train_losses, val_losses = [], []

    print(f"  Training (LOOKBACK={LOOKBACK}, BATCH={BATCH_SIZE}, max {EPOCHS} epochs)...")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_loss = 0.0
        for xp, xf, y in train_loader:
            xp, xf, y = xp.to(DEVICE), xf.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xp, xf), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xp, xf, y in val_loader:
                xp, xf, y = xp.to(DEVICE), xf.to(DEVICE), y.to(DEVICE)
                vl += criterion(model(xp, xf), y).item()

        tl = ep_loss / len(train_loader)
        vl = vl   / len(val_loader)
        train_losses.append(tl)
        val_losses.append(vl)
        scheduler.step()

        lr_now = optimizer.param_groups[0]['lr']
        print(f"    Epoch {epoch:3d}/{EPOCHS}  "
              f"train={tl:.4f}  val={vl:.4f}  lr={lr_now:.6f}")

        if vl < best_val:
            best_val = vl; pat_count = 0
            torch.save(model.state_dict(),
                       f'outputs_cnnlstm/cnnlstm_{mkt}_best.pt')
        else:
            pat_count += 1
            if pat_count >= PATIENCE:
                print(f"    Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(
        f'outputs_cnnlstm/cnnlstm_{mkt}_best.pt', map_location=DEVICE))

    model.eval()
    preds_log, targets_log = [], []
    with torch.no_grad():
        for xp, xf, y in test_loader:
            xp, xf = xp.to(DEVICE), xf.to(DEVICE)
            preds_log.append(model(xp, xf).cpu().numpy())
            targets_log.append(y.numpy())

    preds_price  = np.clip(np.expm1(np.concatenate(preds_log).flatten()),  0, 20000)
    target_price = np.clip(np.expm1(np.concatenate(targets_log).flatten()), 0, 20000)

    print(f"  Results:")
    m = compute_metrics(target_price, preds_price, 'Test')
    all_metrics[mkt] = m

    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.plot(train_losses, color='#e74c3c', lw=1.2, label='Train Loss')
    ax.plot(val_losses,   color='#27ae60', lw=1.2, label='Val Loss')
    ax.axvline(int(np.argmin(val_losses)), color='yellow',
               lw=1.5, ls='--', label=f'Best epoch')
    ax.set_title(f'CNN-LSTM Training Curve — {mkt}', color='white')
    ax.set_xlabel('Epoch', color='#aaa'); ax.set_ylabel('Huber Loss', color='#aaa')
    ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_cnnlstm/cnnlstm_loss_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    save_pred_plot(target_price, preds_price,
                   f'CNN-LSTM — {mkt}',
                   f'outputs_cnnlstm/cnnlstm_pred_{mkt}.png')

    with open(f'outputs_cnnlstm/cnnlstm_scaler_{mkt}.pkl', 'wb') as f:
        pickle.dump({'scaler': scaler, 'past_features': PAST_FEATURES,
                     'future_features': FUTURE_FEATURES,
                     'lookback': LOOKBACK, 'horizon': HORIZON}, f)
    print(f"  Saved: outputs_cnnlstm/cnnlstm_{mkt}_best.pt")


print(f"\n{'='*60}")
print("SUMMARY — CNN-LSTM")
print('=' * 60)
rows = [{'Market': mkt, 'MAE': round(m['mae'], 1),
         'RMSE': round(m['rmse'], 1), 'MAPE%': round(m['mape'], 2)}
        for mkt, m in all_metrics.items()]
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('outputs_cnnlstm/metrics.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('CNN-LSTM — Test MAE by Market', color='white', fontsize=14)
for ax, mkt in zip(axes, MARKETS):
    ax.set_facecolor('#1a1d27')
    if mkt in all_metrics:
        mae = all_metrics[mkt]['mae']
        ax.bar([mkt], [mae], color='#2980b9', alpha=0.85)
        ax.text(0, mae + 5, f'Rs.{mae:.0f}', ha='center',
                color='white', fontsize=12, fontweight='bold')
    ax.set_title(mkt, color='white')
    ax.set_ylabel('MAE (Rs./MWh)', color='#aaa')
    ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('outputs_cnnlstm/summary_chart.png',
            dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()

print(f"\nDone! Check outputs_cnnlstm/")
print(f"Device used: {DEVICE}")

MODEL: CNN-LSTM (Speed Optimized)
  Device: MPS (Apple Silicon GPU)
Loading data from final_merged_data.xlsx...

──────────────────────────────────────────────────
Market: GDAM
──────────────────────────────────────────────────
  Train: 102,721 | Val: 26,208 | Test: 14,591
  Samples — Train: 102,530 | Val: 26,017 | Test: 14,400
  Parameters: 1,634,273
  Training (LOOKBACK=96, BATCH=128, max 50 epochs)...
    Epoch   1/50  train=11.7096  val=11.3169  lr=0.000333
    Epoch   2/50  train=0.5414  val=0.0920  lr=0.000667
    Epoch   3/50  train=0.1363  val=0.0694  lr=0.001000
    Epoch   4/50  train=0.1040  val=0.0739  lr=0.000999
    Epoch   5/50  train=0.0834  val=0.0646  lr=0.000996
    Epoch   6/50  train=0.0712  val=0.0490  lr=0.000990
    Epoch   7/50  train=0.0631  val=0.0507  lr=0.000982
    Epoch   8/50  train=0.0588  val=0.0440  lr=0.000972
    Epoch   9/50  train=0.0563  val=0.0439  lr=0.000960
    Epoch  10/50  train=0.0545  val=0.0471  lr=0.000946
    Epoch  11/50  train=0.0526